# 第 34 章：Transformer、注意力与科学基础模型概念

这个 notebook 用一个极小的 token 化光谱序列数据集，演示为什么注意力会成为现代序列模型的关键机制：

- 先用 bag-of-tokens 建立一个顺序信息全丢失的 baseline
- 再实现一个最小位置感知 attention 头
- 比较两者在同一组 token、不同顺序上下文上的表现
- 再接一个 masked-token workflow，理解最小预训练目标与人工复核队列
- 最后把这个教学例子连到 Transformer、masked token 和科学基础模型的直觉上

教学说明：这里默认不依赖 NumPy、PyTorch 或大型模型库；如果环境里安装了 `torch`，最后一格会给出同一个 attention 计算的张量版本。


In [ ]:
from __future__ import annotations

import csv
from collections import Counter
from pathlib import Path

DATA_PATH = Path("../../data/small/spectral_token_attention_demo.csv").resolve()
SEQUENCE_LENGTH = 6

with DATA_PATH.open(newline="", encoding="utf-8") as handle:
    rows = list(csv.DictReader(handle))

train_rows = [row for row in rows if row["split"] == "train"]
test_rows = [row for row in rows if row["split"] == "test"]


def tokens_from_row(row):
    return [row[f"tok{index:02d}"] for index in range(SEQUENCE_LENGTH)]


print(f"Loaded {len(rows)} tokenized sequences from {DATA_PATH.name}")
print("class counts:", dict(Counter(row["sequence_label"] for row in rows)))
print("train/test sizes:", len(train_rows), len(test_rows))
print("Representative sequences:")
for sample_id in ["E01", "A01", "R01"]:
    row = next(item for item in rows if item["sample_id"] == sample_id)
    print(sample_id, row["sequence_label"], tokens_from_row(row))


In [ ]:
import math

vocabulary = sorted({token for row in rows for token in tokens_from_row(row)})
label_order = sorted({row["sequence_label"] for row in rows})


def bag_of_tokens(row):
    counts = Counter(tokens_from_row(row))
    return [counts[token] for token in vocabulary]


def centroid(vectors):
    return [sum(values) / len(values) for values in zip(*vectors)]


def euclidean_distance(left, right):
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(left, right)))


def nearest_centroid(train_items, feature_fn, test_items):
    centers = {}
    for label in label_order:
        vectors = [feature_fn(item) for item in train_items if item["sequence_label"] == label]
        centers[label] = centroid(vectors)

    predictions = []
    for item in test_items:
        vector = feature_fn(item)
        ranked = sorted(
            ((label, euclidean_distance(vector, centers[label])) for label in label_order),
            key=lambda pair: pair[1],
        )
        predictions.append({
            "sample_id": item["sample_id"],
            "true_label": item["sequence_label"],
            "predicted_label": ranked[0][0],
            "distances": {label: round(distance, 3) for label, distance in ranked},
        })
    return centers, predictions


def accuracy(predictions):
    return sum(item["true_label"] == item["predicted_label"] for item in predictions) / len(predictions)


def confusion(predictions):
    matrix = {}
    for item in predictions:
        truth = item["true_label"]
        predicted = item["predicted_label"]
        matrix.setdefault(truth, {})
        matrix[truth][predicted] = matrix[truth].get(predicted, 0) + 1
    return matrix


bag_centers, bag_predictions = nearest_centroid(train_rows, bag_of_tokens, test_rows)

print("Bag-of-tokens nearest-centroid baseline:")
print("class centroids:", bag_centers)
for item in bag_predictions:
    print(item["sample_id"], item["true_label"], "->", item["predicted_label"], item["distances"])
print("bag accuracy:", round(accuracy(bag_predictions), 3))
print("bag confusion matrix:", confusion(bag_predictions))


In [ ]:
value_map = {
    "sky_residual": [1.0, 0.0, 0.0],
    "balmer_absorption": [0.0, 1.0, 0.0],
    "ha_emission": [0.0, 0.0, 1.0],
    "blue_continuum": [0.0, 0.0, 0.0],
    "red_continuum": [0.0, 0.0, 0.0],
    "center_mark": [0.0, 0.0, 0.0],
}
context_index_to_label = ["artifact_context", "absorption_context", "emission_context"]


def softmax(values):
    shifted = [value - max(values) for value in values]
    exp_values = [math.exp(value) for value in shifted]
    total = sum(exp_values)
    return [value / total for value in exp_values]


def attention_summary(row, focus_strength=4.0):
    tokens = tokens_from_row(row)
    center_index = tokens.index("center_mark")

    # The notebook uses a tiny teaching head: the query asks for the token right after the center marker.
    query = 1.0
    keys = [focus_strength if index == center_index + 1 else 0.0 for index in range(len(tokens))]
    scores = [query * key for key in keys]
    weights = softmax(scores)

    context = [0.0, 0.0, 0.0]
    for weight, token in zip(weights, tokens):
        value = value_map[token]
        context = [current + weight * token_value for current, token_value in zip(context, value)]

    predicted_label = context_index_to_label[max(range(len(context)), key=lambda index: context[index])]
    return {
        "tokens": tokens,
        "scores": scores,
        "weights": weights,
        "context": context,
        "predicted_label": predicted_label,
    }


attention_predictions = []
for row in test_rows:
    summary = attention_summary(row)
    attention_predictions.append({
        "sample_id": row["sample_id"],
        "true_label": row["sequence_label"],
        "predicted_label": summary["predicted_label"],
        "context": [round(value, 3) for value in summary["context"]],
    })

print("Position-aware attention head:")
for item in attention_predictions:
    print(item["sample_id"], item["true_label"], "->", item["predicted_label"], item["context"])
print("attention accuracy:", round(accuracy(attention_predictions), 3))
print("attention confusion matrix:", confusion(attention_predictions))
print("accuracy improvement over bag baseline:", round(accuracy(attention_predictions) - accuracy(bag_predictions), 3))


In [ ]:
label_to_token = {
    "emission_context": "ha_emission",
    "absorption_context": "balmer_absorption",
    "artifact_context": "sky_residual",
}

print("Attention weights on representative test samples:")
for sample_id in ["E04", "A04", "R04"]:
    row = next(item for item in rows if item["sample_id"] == sample_id)
    summary = attention_summary(row)
    uniform_weights = [1.0 / len(summary["tokens"]) for _ in summary["tokens"]]
    uniform_context = [0.0, 0.0, 0.0]
    for weight, token in zip(uniform_weights, summary["tokens"]):
        value = value_map[token]
        uniform_context = [current + weight * token_value for current, token_value in zip(uniform_context, value)]

    actual_token = summary["tokens"][summary["tokens"].index("center_mark") + 1]
    guessed_token = label_to_token[summary["predicted_label"]]

    print()
    print(sample_id, row["sequence_label"])
    print("tokens =", summary["tokens"])
    print("attention weights =", [round(value, 3) for value in summary["weights"]])
    print("uniform context =", [round(value, 3) for value in uniform_context])
    print("attention context =", [round(value, 3) for value in summary["context"]])
    print("masked-token guess =", guessed_token, "actual token =", actual_token)

print("Interpretation:")
print("  When we remove order and only keep token counts, all three classes collapse to the same representation.")
print("  Once the query is allowed to read the token right after center_mark, the class becomes obvious.")
print("  That is the key Transformer intuition: attention is a conditional readout mechanism, not a uniform average.")


In [ ]:
WORKFLOW_DATA_PATH = Path("../../data/small/spectral_masked_token_workflow_demo.csv").resolve()
WORKFLOW_SEQUENCE_LENGTH = 8
WORKFLOW_MASK_INDEX = 3
WORKFLOW_SEED = 3
WORKFLOW_D_MODEL = 5
WORKFLOW_EPOCHS = 50
WORKFLOW_LEARNING_RATE = 0.03

with WORKFLOW_DATA_PATH.open(newline="", encoding="utf-8") as handle:
    workflow_rows = list(csv.DictReader(handle))

workflow_train_rows = [row for row in workflow_rows if row["split_role"] == "train"]
workflow_validation_rows = [row for row in workflow_rows if row["split_role"] == "validation"]
workflow_test_rows = [row for row in workflow_rows if row["split_role"] == "test"]
workflow_review_rows = [row for row in workflow_rows if row["split_role"] == "review"]
workflow_target_order = sorted({row["target_token"] for row in workflow_rows if row["quality_flag"] == "clean"})
workflow_target_to_index = {token: index for index, token in enumerate(workflow_target_order)}
workflow_vocabulary = sorted({token for row in workflow_rows for token in [row[f"tok{index:02d}"] for index in range(WORKFLOW_SEQUENCE_LENGTH)]})
workflow_token_to_index = {token: index for index, token in enumerate(workflow_vocabulary)}


def workflow_tokens_from_row(row):
    return [row[f"tok{index:02d}"] for index in range(WORKFLOW_SEQUENCE_LENGTH)]


print(f"Loaded {len(workflow_rows)} masked-token sequences from {WORKFLOW_DATA_PATH.name}")
print("split-role counts:", dict(Counter(row["split_role"] for row in workflow_rows)))
print("quality counts:", dict(Counter(row["quality_flag"] for row in workflow_rows)))
print("Representative masked sequences:")
for sample_id in ["T_em0", "X_ab0", "R_blend"]:
    row = next(item for item in workflow_rows if item["sample_id"] == sample_id)
    print(sample_id, row["split_role"], row["quality_flag"], row["target_token"], workflow_tokens_from_row(row))


def workflow_bag_of_observed_tokens(row):
    counts = Counter(workflow_tokens_from_row(row))
    return [counts[token] for token in workflow_vocabulary]


def workflow_nearest_centroid(train_items, feature_fn, test_items):
    centers = {}
    for label in workflow_target_order:
        vectors = [feature_fn(item) for item in train_items if item["target_token"] == label]
        centers[label] = centroid(vectors)

    predictions = []
    for item in test_items:
        vector = feature_fn(item)
        ranked = sorted(
            ((label, euclidean_distance(vector, centers[label])) for label in workflow_target_order),
            key=lambda pair: pair[1],
        )
        predictions.append(
            {
                "sample_id": item["sample_id"],
                "true_label": item["target_token"],
                "predicted_label": ranked[0][0],
                "distances": {label: round(distance, 3) for label, distance in ranked},
            }
        )
    return centers, predictions


def workflow_route_from_confidence(confidence, ready_threshold):
    return "masked_prediction_ready" if confidence >= ready_threshold else "manual_review"


def workflow_cross_entropy(probabilities, target_token):
    target_index = workflow_target_to_index[target_token]
    return -math.log(max(probabilities[target_index], 1e-12))


workflow_bag_centers, workflow_bag_predictions = workflow_nearest_centroid(
    workflow_train_rows,
    workflow_bag_of_observed_tokens,
    workflow_test_rows,
)

print()
print("Masked-token baseline: bag-of-observed tokens on clean test")
print("class centroids:", workflow_bag_centers)
for item in workflow_bag_predictions:
    print(item["sample_id"], item["true_label"], "->", item["predicted_label"], item["distances"])
print("baseline accuracy:", round(accuracy(workflow_bag_predictions), 3))
print("baseline confusion matrix:", confusion(workflow_bag_predictions))

workflow_value_map = {
    "balmer_absorption": [4.0, 0.0, 0.0],
    "ha_emission": [0.0, 4.0, 0.0],
    "sky_residual": [0.0, 0.0, 4.0],
    "blue_continuum": [0.0, 0.0, 0.0],
    "red_continuum": [0.0, 0.0, 0.0],
    "bridge_token": [0.0, 0.0, 0.0],
    "center_mark": [0.0, 0.0, 0.0],
    "[MASK]": [0.0, 0.0, 0.0],
    "blend_candidate": [0.0, 0.0, 0.0],
}


def workflow_masked_summary(row, focus_strength=4.5):
    tokens = workflow_tokens_from_row(row)
    center_index = tokens.index("center_mark")
    source_index = center_index + 4
    scores = [focus_strength if index == source_index else 0.0 for index in range(len(tokens))]
    weights = softmax(scores)

    logits = [0.0, 0.0, 0.0]
    for weight, token in zip(weights, tokens):
        value = workflow_value_map.get(token, [0.0, 0.0, 0.0])
        logits = [current + weight * token_value for current, token_value in zip(logits, value)]

    probabilities = softmax(logits)
    predicted_target = workflow_target_order[max(range(len(probabilities)), key=lambda index: probabilities[index])]
    return {
        "tokens": tokens,
        "scores": scores,
        "weights": weights,
        "logits": logits,
        "probabilities": probabilities,
        "predicted_target": predicted_target,
        "confidence": max(probabilities),
        "source_index": source_index,
        "source_token": tokens[source_index],
    }


workflow_handcrafted_validation = []
for row in workflow_validation_rows:
    summary = workflow_masked_summary(row)
    workflow_handcrafted_validation.append(
        {
            "sample_id": row["sample_id"],
            "true_label": row["target_token"],
            "predicted_label": summary["predicted_target"],
            "confidence": summary["confidence"],
        }
    )

workflow_handcrafted_ready_threshold = min(item["confidence"] for item in workflow_handcrafted_validation) - 0.02
workflow_handcrafted_test_predictions = []
print()
print("Hand-crafted position-aware masked-token head on clean test")
for row in workflow_test_rows:
    summary = workflow_masked_summary(row)
    workflow_handcrafted_test_predictions.append(
        {
            "sample_id": row["sample_id"],
            "true_label": row["target_token"],
            "predicted_label": summary["predicted_target"],
        }
    )
    print(
        row["sample_id"],
        row["target_token"],
        "->",
        summary["predicted_target"],
        [round(value, 3) for value in summary["probabilities"]],
        "source=",
        summary["source_token"],
    )
print("hand-crafted accuracy:", round(accuracy(workflow_handcrafted_test_predictions), 3))
print("hand-crafted confusion matrix:", confusion(workflow_handcrafted_test_predictions))
print("accuracy improvement over mask-blind baseline:", round(accuracy(workflow_handcrafted_test_predictions) - accuracy(workflow_bag_predictions), 3))
print("hand-crafted ready threshold =", round(workflow_handcrafted_ready_threshold, 3))

workflow_handcrafted_routes = []
for row in workflow_test_rows + workflow_review_rows:
    summary = workflow_masked_summary(row)
    route = workflow_route_from_confidence(summary["confidence"], workflow_handcrafted_ready_threshold)
    workflow_handcrafted_routes.append(
        {
            "sample_id": row["sample_id"],
            "quality_flag": row["quality_flag"],
            "target_token": row["target_token"],
            "predicted_target": summary["predicted_target"],
            "confidence": summary["confidence"],
            "route": route,
            "source_token": summary["source_token"],
        }
    )

print("hand-crafted route counts:", dict(Counter(item["route"] for item in workflow_handcrafted_routes)))
print("hand-crafted manual review ids:", [item["sample_id"] for item in workflow_handcrafted_routes if item["route"] == "manual_review"])

import random


def zeros_vector(size):
    return [0.0] * size


def zeros_matrix(num_rows, num_cols):
    return [[0.0] * num_cols for _ in range(num_rows)]


def zero_like(value):
    if isinstance(value[0], list):
        return zeros_matrix(len(value), len(value[0]))
    return zeros_vector(len(value))


def init_matrix(rng, num_rows, num_cols, scale):
    return [[rng.uniform(-scale, scale) for _ in range(num_cols)] for _ in range(num_rows)]


def dot_product(left, right):
    return sum(left_value * right_value for left_value, right_value in zip(left, right))


def matrix_vector(matrix, vector):
    return [dot_product(row, vector) for row in matrix]


def transpose_matrix_vector(matrix, vector):
    output = [0.0] * len(matrix[0])
    for row_index, row in enumerate(matrix):
        for column_index, value in enumerate(row):
            output[column_index] += value * vector[row_index]
    return output


def outer_add(target, left, right):
    for row_index, left_value in enumerate(left):
        row = target[row_index]
        for column_index, right_value in enumerate(right):
            row[column_index] += left_value * right_value


def argmax(values):
    return max(range(len(values)), key=lambda index: values[index])


class TinySelfAttentionMaskedModel:
    parameter_names = ["token_embeddings", "position_embeddings", "Wq", "Wk", "Wv", "output_bias"]

    def __init__(self, seed=WORKFLOW_SEED, d_model=WORKFLOW_D_MODEL):
        rng = random.Random(seed)
        self.d_model = d_model
        self.token_embeddings = init_matrix(rng, len(workflow_vocabulary), d_model, scale=0.30)
        self.position_embeddings = init_matrix(rng, WORKFLOW_SEQUENCE_LENGTH, d_model, scale=0.20)
        self.Wq = init_matrix(rng, d_model, d_model, scale=0.20)
        self.Wk = init_matrix(rng, d_model, d_model, scale=0.20)
        self.Wv = init_matrix(rng, len(workflow_target_order), d_model, scale=0.20)
        self.output_bias = zeros_vector(len(workflow_target_order))
        self.scale = math.sqrt(d_model)

    def forward(self, row):
        tokens = workflow_tokens_from_row(row)
        states = []
        for position, token in enumerate(tokens):
            token_embedding = self.token_embeddings[workflow_token_to_index[token]]
            position_embedding = self.position_embeddings[position]
            states.append(
                [token_value + position_value for token_value, position_value in zip(token_embedding, position_embedding)]
            )

        query = matrix_vector(self.Wq, states[WORKFLOW_MASK_INDEX])
        keys = [matrix_vector(self.Wk, state) for state in states]
        attention_scores = [dot_product(query, key) / self.scale for key in keys]
        attention_weights = softmax(attention_scores)

        value_logits = [matrix_vector(self.Wv, state) for state in states]
        logits = self.output_bias[:]
        for weight, token_logits in zip(attention_weights, value_logits):
            logits = [current + weight * contribution for current, contribution in zip(logits, token_logits)]

        probabilities = softmax(logits)
        peak_index = argmax(attention_weights)
        predicted_target = workflow_target_order[argmax(probabilities)]
        return {
            "tokens": tokens,
            "states": states,
            "query": query,
            "keys": keys,
            "attention_scores": attention_scores,
            "attention_weights": attention_weights,
            "value_logits": value_logits,
            "logits": logits,
            "probabilities": probabilities,
            "predicted_target": predicted_target,
            "confidence": max(probabilities),
            "peak_index": peak_index,
            "peak_token": tokens[peak_index],
        }

    def loss_and_gradients(self, row):
        summary = self.forward(row)
        target_index = workflow_target_to_index[row["target_token"]]
        loss = workflow_cross_entropy(summary["probabilities"], row["target_token"])

        gradients = {
            "token_embeddings": zeros_matrix(len(workflow_vocabulary), self.d_model),
            "position_embeddings": zeros_matrix(WORKFLOW_SEQUENCE_LENGTH, self.d_model),
            "Wq": zeros_matrix(self.d_model, self.d_model),
            "Wk": zeros_matrix(self.d_model, self.d_model),
            "Wv": zeros_matrix(len(workflow_target_order), self.d_model),
            "output_bias": zeros_vector(len(workflow_target_order)),
        }

        d_logits = summary["probabilities"][:]
        d_logits[target_index] -= 1.0
        for index, value in enumerate(d_logits):
            gradients["output_bias"][index] += value

        d_value_logits = [zeros_vector(len(workflow_target_order)) for _ in range(WORKFLOW_SEQUENCE_LENGTH)]
        d_attention_weights = zeros_vector(WORKFLOW_SEQUENCE_LENGTH)
        for position in range(WORKFLOW_SEQUENCE_LENGTH):
            d_attention_weights[position] = dot_product(d_logits, summary["value_logits"][position])
            for class_index, value in enumerate(d_logits):
                d_value_logits[position][class_index] += summary["attention_weights"][position] * value

        attention_weight_average = sum(
            gradient * weight for gradient, weight in zip(d_attention_weights, summary["attention_weights"])
        )
        d_attention_scores = [
            weight * (gradient - attention_weight_average)
            for gradient, weight in zip(d_attention_weights, summary["attention_weights"])
        ]

        d_query = zeros_vector(self.d_model)
        d_keys = [zeros_vector(self.d_model) for _ in range(WORKFLOW_SEQUENCE_LENGTH)]
        for position in range(WORKFLOW_SEQUENCE_LENGTH):
            coefficient = d_attention_scores[position] / self.scale
            for dimension in range(self.d_model):
                d_query[dimension] += coefficient * summary["keys"][position][dimension]
                d_keys[position][dimension] += coefficient * summary["query"][dimension]

        d_states = [zeros_vector(self.d_model) for _ in range(WORKFLOW_SEQUENCE_LENGTH)]
        outer_add(gradients["Wq"], d_query, summary["states"][WORKFLOW_MASK_INDEX])
        query_state_gradient = transpose_matrix_vector(self.Wq, d_query)
        for dimension, value in enumerate(query_state_gradient):
            d_states[WORKFLOW_MASK_INDEX][dimension] += value

        for position in range(WORKFLOW_SEQUENCE_LENGTH):
            outer_add(gradients["Wk"], d_keys[position], summary["states"][position])
            key_state_gradient = transpose_matrix_vector(self.Wk, d_keys[position])
            outer_add(gradients["Wv"], d_value_logits[position], summary["states"][position])
            value_state_gradient = transpose_matrix_vector(self.Wv, d_value_logits[position])
            for dimension in range(self.d_model):
                d_states[position][dimension] += key_state_gradient[dimension] + value_state_gradient[dimension]

        for position, token in enumerate(summary["tokens"]):
            token_index = workflow_token_to_index[token]
            for dimension, value in enumerate(d_states[position]):
                gradients["token_embeddings"][token_index][dimension] += value
                gradients["position_embeddings"][position][dimension] += value

        return loss, gradients


class AdamOptimizer:
    def __init__(self, model, learning_rate=WORKFLOW_LEARNING_RATE):
        self.learning_rate = learning_rate
        self.beta1 = 0.9
        self.beta2 = 0.999
        self.epsilon = 1e-8
        self.step_count = 0
        self.first_moments = {}
        self.second_moments = {}
        for name in model.parameter_names:
            parameter = getattr(model, name)
            self.first_moments[name] = zero_like(parameter)
            self.second_moments[name] = zero_like(parameter)

    def step(self, model, gradients):
        self.step_count += 1
        for name in model.parameter_names:
            parameter = getattr(model, name)
            gradient = gradients[name]
            first_moment = self.first_moments[name]
            second_moment = self.second_moments[name]
            if isinstance(parameter[0], list):
                for row_index, row in enumerate(parameter):
                    for column_index, _ in enumerate(row):
                        grad_value = gradient[row_index][column_index]
                        first_moment[row_index][column_index] = (
                            self.beta1 * first_moment[row_index][column_index] + (1.0 - self.beta1) * grad_value
                        )
                        second_moment[row_index][column_index] = (
                            self.beta2 * second_moment[row_index][column_index] + (1.0 - self.beta2) * grad_value * grad_value
                        )
                        first_unbiased = first_moment[row_index][column_index] / (1.0 - self.beta1**self.step_count)
                        second_unbiased = second_moment[row_index][column_index] / (1.0 - self.beta2**self.step_count)
                        row[column_index] -= self.learning_rate * first_unbiased / (math.sqrt(second_unbiased) + self.epsilon)
            else:
                for index, _ in enumerate(parameter):
                    grad_value = gradient[index]
                    first_moment[index] = self.beta1 * first_moment[index] + (1.0 - self.beta1) * grad_value
                    second_moment[index] = self.beta2 * second_moment[index] + (1.0 - self.beta2) * grad_value * grad_value
                    first_unbiased = first_moment[index] / (1.0 - self.beta1**self.step_count)
                    second_unbiased = second_moment[index] / (1.0 - self.beta2**self.step_count)
                    parameter[index] -= self.learning_rate * first_unbiased / (math.sqrt(second_unbiased) + self.epsilon)


def workflow_evaluate_model(model, rows):
    predictions = []
    total_loss = 0.0
    for row in rows:
        summary = model.forward(row)
        total_loss += workflow_cross_entropy(summary["probabilities"], row["target_token"])
        predictions.append(
            {
                "sample_id": row["sample_id"],
                "true_label": row["target_token"],
                "predicted_label": summary["predicted_target"],
                "confidence": summary["confidence"],
                "probabilities": summary["probabilities"],
                "attention_weights": summary["attention_weights"],
                "peak_index": summary["peak_index"],
                "peak_token": summary["peak_token"],
            }
        )
    return predictions, total_loss / len(rows)


tiny_attention_model = TinySelfAttentionMaskedModel()
tiny_attention_optimizer = AdamOptimizer(tiny_attention_model)
training_curve = []
for epoch in range(1, WORKFLOW_EPOCHS + 1):
    epoch_loss = 0.0
    for row in workflow_train_rows:
        loss, gradients = tiny_attention_model.loss_and_gradients(row)
        tiny_attention_optimizer.step(tiny_attention_model, gradients)
        epoch_loss += loss
    if epoch in {1, 10, 20, 30, 40, 50}:
        validation_predictions, validation_loss = workflow_evaluate_model(tiny_attention_model, workflow_validation_rows)
        training_curve.append(
            {
                "epoch": epoch,
                "train_loss": epoch_loss / len(workflow_train_rows),
                "validation_loss": validation_loss,
                "validation_accuracy": accuracy(validation_predictions),
            }
        )

workflow_trainable_validation_predictions, workflow_trainable_validation_loss = workflow_evaluate_model(
    tiny_attention_model,
    workflow_validation_rows,
)
workflow_trainable_test_predictions, workflow_trainable_test_loss = workflow_evaluate_model(
    tiny_attention_model,
    workflow_test_rows,
)
workflow_trainable_ready_threshold = min(item["confidence"] for item in workflow_trainable_validation_predictions) - 0.02

print()
print("Trainable tiny self-attention masked-token learner")
print(
    "training config:",
    {
        "seed": WORKFLOW_SEED,
        "d_model": WORKFLOW_D_MODEL,
        "epochs": WORKFLOW_EPOCHS,
        "learning_rate": WORKFLOW_LEARNING_RATE,
    },
)
print(
    "training curve:",
    [
        (item["epoch"], round(item["train_loss"], 4), round(item["validation_loss"], 4), round(item["validation_accuracy"], 3))
        for item in training_curve
    ],
)
print(
    "validation confidences:",
    [(item["sample_id"], round(item["confidence"], 4)) for item in workflow_trainable_validation_predictions],
)
print("validation loss =", round(workflow_trainable_validation_loss, 4))
print("trainable ready threshold =", round(workflow_trainable_ready_threshold, 3))
print()
print("Trainable learner on clean test")
for item in workflow_trainable_test_predictions:
    print(
        item["sample_id"],
        item["true_label"],
        "->",
        item["predicted_label"],
        [round(value, 4) for value in item["probabilities"]],
        "peak=",
        f"tok{item['peak_index']:02d}",
        item["peak_token"],
    )
print("trainable accuracy:", round(accuracy(workflow_trainable_test_predictions), 3))
print("trainable confusion matrix:", confusion(workflow_trainable_test_predictions))
print("trainable clean-test loss =", round(workflow_trainable_test_loss, 4))

workflow_trainable_routes = []
for row in workflow_test_rows + workflow_review_rows:
    summary = tiny_attention_model.forward(row)
    route = workflow_route_from_confidence(summary["confidence"], workflow_trainable_ready_threshold)
    workflow_trainable_routes.append(
        {
            "sample_id": row["sample_id"],
            "quality_flag": row["quality_flag"],
            "target_token": row["target_token"],
            "predicted_target": summary["predicted_target"],
            "confidence": summary["confidence"],
            "route": route,
            "peak_index": summary["peak_index"],
            "peak_token": summary["peak_token"],
        }
    )

print()
print("Held-out routing summary for trainable learner:")
for item in workflow_trainable_routes:
    print(
        item["sample_id"],
        item["quality_flag"],
        item["target_token"],
        "->",
        item["route"],
        "pred=",
        item["predicted_target"],
        "confidence=",
        round(item["confidence"], 4),
        "peak=",
        f"tok{item['peak_index']:02d}",
        item["peak_token"],
    )
print("route counts:", dict(Counter(item["route"] for item in workflow_trainable_routes)))
print("manual review ids:", [item["sample_id"] for item in workflow_trainable_routes if item["route"] == "manual_review"])
print("masked prediction ready ids:", [item["sample_id"] for item in workflow_trainable_routes if item["route"] == "masked_prediction_ready"])
print(
    "review-queue capture =",
    f"{sum(item['route'] == 'manual_review' for item in workflow_trainable_routes if item['quality_flag'] != 'clean')}/{len(workflow_review_rows)}",
)

print()
print("Representative learned attention snapshots:")
for sample_id in ["X_em0", "X_ab0", "R_blend", "R_mask"]:
    row = next(item for item in workflow_rows if item["sample_id"] == sample_id)
    summary = tiny_attention_model.forward(row)
    route = next(item["route"] for item in workflow_trainable_routes if item["sample_id"] == sample_id)
    print()
    print(sample_id, row["quality_flag"], row["target_token"])
    print("tokens =", summary["tokens"])
    print("attention weights =", [round(value, 3) for value in summary["attention_weights"]])
    print("token logits at peak =", [round(value, 3) for value in summary["value_logits"][summary["peak_index"]]])
    print("final logits =", [round(value, 3) for value in summary["logits"]])
    print("token probabilities =", [round(value, 4) for value in summary["probabilities"]])
    print("predicted token =", summary["predicted_target"], "peak =", f"tok{summary['peak_index']:02d}", summary["peak_token"], "route =", route)

print("Interpretation:")
print("  The mask-blind baseline still collapses to 0.333 because every clean row shows the same observed token multiset.")
print("  The hand-crafted head hard-codes a jump to tok06, while the trainable learner starts from random token/position embeddings and learns the same long-range readout from six clean training rows.")
print("  On the clean validation/test rows the learned attention becomes nearly one-hot at tok06, which is why the tiny model reaches 1.000 accuracy without any large-model library.")
print("  The low-SNR blend token and second mask rows stay far below the validation-calibrated confidence threshold, so the workflow still routes both held-out anomalies to manual review.")


In [ ]:
try:
    import matplotlib.pyplot as plt

    sample_ids = ["X_em0", "R_blend"]
    figure, axes = plt.subplots(1, len(sample_ids), figsize=(9.0, 3.4), sharey=True)
    if len(sample_ids) == 1:
        axes = [axes]

    for axis, sample_id in zip(axes, sample_ids):
        row = next(item for item in workflow_rows if item["sample_id"] == sample_id)
        summary = tiny_attention_model.forward(row)
        positions = list(range(WORKFLOW_SEQUENCE_LENGTH))
        axis.bar(positions, summary["attention_weights"], color="#2f6b99")
        axis.set_xticks(positions, summary["tokens"], rotation=25, ha="right")
        axis.set_title(f"{sample_id}: peak tok{summary['peak_index']:02d}")
        axis.set_ylim(0, 1.0)
        axis.grid(axis="y", alpha=0.3)
    axes[0].set_ylabel("attention weight")
    figure.suptitle("Learned masked-token attention")
    figure.tight_layout()
    plt.show()
except Exception as exc:
    print(f"Optional matplotlib plot skipped: {exc}")


In [ ]:
try:
    import torch

    row = next(item for item in workflow_rows if item["sample_id"] == "X_em0")
    token_indices = torch.tensor(
        [workflow_token_to_index[token] for token in workflow_tokens_from_row(row)],
        dtype=torch.long,
    )
    position_indices = torch.arange(WORKFLOW_SEQUENCE_LENGTH, dtype=torch.long)

    token_embeddings = torch.tensor(tiny_attention_model.token_embeddings, dtype=torch.float32)
    position_embeddings = torch.tensor(tiny_attention_model.position_embeddings, dtype=torch.float32)
    Wq = torch.tensor(tiny_attention_model.Wq, dtype=torch.float32)
    Wk = torch.tensor(tiny_attention_model.Wk, dtype=torch.float32)
    Wv = torch.tensor(tiny_attention_model.Wv, dtype=torch.float32)
    output_bias = torch.tensor(tiny_attention_model.output_bias, dtype=torch.float32)

    states = token_embeddings[token_indices] + position_embeddings[position_indices]
    query = torch.matmul(Wq, states[WORKFLOW_MASK_INDEX])
    keys = torch.matmul(states, Wk.T)
    attention_scores = torch.matmul(keys, query) / math.sqrt(tiny_attention_model.d_model)
    attention_weights = torch.softmax(attention_scores, dim=0)
    value_logits = torch.matmul(states, Wv.T)
    logits = output_bias + torch.sum(attention_weights.unsqueeze(-1) * value_logits, dim=0)
    probabilities = torch.softmax(logits, dim=0)

    print("Optional torch learned weights:", [round(float(value), 3) for value in attention_weights])
    print("Optional torch learned probabilities:", [round(float(value), 4) for value in probabilities])
except Exception as exc:
    print(f"Optional PyTorch comparison skipped: {exc}")
